# Forest Reconstruction Pipeline
Full pipeline: point cloud segmentation (PointTree) -> tree reconstruction (AdTree)

**Input:** single point cloud file (.las, .laz, .ply, .txt, .csv)

**Output:**
- Per tree: individual point cloud (.laz), skeleton (.ply), branches (.obj), leaves (.obj, filtered versions)
- Forest: labeled full cloud, merged branches, merged leaves, assembly JSON, summary CSV
- Final ZIP with all results

**Required files to upload:**
- Your point cloud file (.las / .laz / .ply / .txt / .csv)
- `AdTree-main.zip`


## Step 1: Install dependencies
> Run this cell first. The kernel will restart automatically afterwards.

In [ ]:
import subprocess, sys

# Pin numpy to compatible version before everything else
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2.0'], check=True)

# PyTorch CPU (maximum Colab compatibility)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.5.0', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)

# torch-scatter and torch-cluster
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch-scatter', 'torch-cluster',
    '-f', 'https://data.pyg.org/whl/torch-2.5.0+cpu.html'], check=True)

# pointtree, pointtorch and I/O libs
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pointtree', 'pointtorch', 'laspy[lazrs]', 'open3d', 'scipy'], check=True)

print('Installation complete. Kernel will now restart...')

import IPython
IPython.Application.instance().kernel.do_shutdown(True)


## Step 2: System dependencies + compile AdTree
> Run after kernel restart. Takes 3-5 minutes.

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq \
    cmake build-essential \
    libboost-all-dev \
    libgl1-mesa-dev libglu1-mesa-dev \
    libxrandr-dev libxinerama-dev libxcursor-dev libxi-dev libxext-dev

rm -rf /content/AdTree_build/
mkdir -p /content/AdTree_build/
unzip /content/AdTree-main.zip -d /content/AdTree_build/ 2>&1 | tail -3

SRCDIR=/content/AdTree_build/AdTree-main

# OpenGL fix
find $SRCDIR -name "CMakeLists.txt" -exec \
    sed -i 's/OpenGL::OpenGL/OpenGL::GL/g' {} \;

sed -i 's/target_link_libraries(${PROJECT_NAME} PRIVATE easy3d_algo/target_link_libraries(${PROJECT_NAME} PRIVATE GLdispatch easy3d_algo/' \
    $SRCDIR/AdTree/CMakeLists.txt
ln -sf /usr/lib/x86_64-linux-gnu/libGLdispatch.so.0 \
       /usr/lib/x86_64-linux-gnu/libGLdispatch.so 2>/dev/null || true

# Apply all patches to skeleton.cpp
python3 << 'PYEOF'
SKEL = '/content/AdTree_build/AdTree-main/AdTree/skeleton.cpp'
with open(SKEL, 'r') as f:
    content = f.read()

# Patch A: leaf density and size
content = content.replace(
    'int density = ceil(random_float() * 10);',
    'int density = ceil(random_float() * 1);'
)
content = content.replace(
    'generate_leaves(currentLeafVertex, 0.05);',
    'generate_leaves(currentLeafVertex, 0.02);'
)
content = content.replace(
    'double radius = 0.2 / log((float)num_edges(simplified_skeleton_));',
    'double radius = 0.04 / log((float)num_edges(simplified_skeleton_));'
)
print('Patch A: leaf density reduced')

# Patch epsiony: 2% -> 10%
content = content.replace(
    '\tdouble epsiony = 0.02;\n',
    '\tdouble epsiony = 0.10;\n'
)
print('Patch epsiony: trunk list at 10%')

# Patch C: leaf base attached to branch tip
old_generate = (
    'void Skeleton::generate_leaves(SGraphVertexDescriptor i_LeafVertex, double leafsize_Factor)\n'
    '{\n'
    '\t//generate a random density number\n'
    '    int density = ceil(random_float() * 1);\n'
    '    double radius = 0.04 / log((float)num_edges(simplified_skeleton_));\n'
    '\t//get the position of the current leaf vertex and its parent\n'
    '    vec3 pCurrent = simplified_skeleton_[i_LeafVertex].cVert;\n'
    '    SGraphVertexDescriptor i_LeafParent = simplified_skeleton_[i_LeafVertex].nParent;\n'
    '    vec3 pParent = simplified_skeleton_[i_LeafParent].cVert;\n'
    '\t//get the end position where the leaf should grow\n'
    '    vec3 pEnd = pCurrent - (random_float() / 2.0) * ((pCurrent - pParent).normalize());\n'
    '\n'
    '\t//generate i-th random leaf\n'
    '\tfor (int i = 0; i < density; ++i)\n'
    '\t{\n'
    '\t\t//generate a random leaf position\n'
    '        vec3 dirLeaf((random_float() - 0.5) / 0.5, (random_float() - 0.5) / 0.5, (random_float() - 0.5) / 0.5);\n'
    '\t\tdirLeaf = dirLeaf.normalize();\n'
    '        double l = random_float() * radius;\n'
    '\t\tvec3 pLeaf = pEnd + dirLeaf * l;\n'
    '\t\t//generate normal and color vector\n'
    '\t\tvec3 dirParent2Leaf = (pLeaf - pParent).normalize();\n'
    '\t\tvec3 normal = (cross(dirParent2Leaf, dirLeaf)).normalize();\n'
    '\t\t//generate a new leaf\n'
    '\t\tLeaf newleaf;\n'
    '\t\tnewleaf.cPos = pLeaf;\n'
    '\t\tnewleaf.cDir = dirLeaf;\n'
    '\t\t//generate a random normal vector direction\n'
    '        vec3 delta((random_float() - 0.5) / 0.5, (random_float() - 0.5) / 0.5, (random_float() - 0.5) / 0.5);\n'
    '        newleaf.cNormal = (normal + random_float()*delta*0.5).normalize();\n'
    '\t\tnewleaf.pSource = i_LeafVertex;\n'
    '\t\tnewleaf.nLength = BoundingDistance_ * leafsize_Factor;\n'
    '\t\tnewleaf.nRad = newleaf.nLength / 5;\n'
    '\t\tVecLeaves_.push_back(newleaf);\n'
    '\t}\n'
    '\n'
    '\treturn;\n'
    '}'
)
new_generate = (
    'void Skeleton::generate_leaves(SGraphVertexDescriptor i_LeafVertex, double leafsize_Factor)\n'
    '{\n'
    '\t//generate a random density number\n'
    '    int density = ceil(random_float() * 1);\n'
    '    double radius = 0.04 / log((float)num_edges(simplified_skeleton_));\n'
    '\t//get the position of the current leaf vertex and its parent\n'
    '    vec3 pCurrent = simplified_skeleton_[i_LeafVertex].cVert;\n'
    '    SGraphVertexDescriptor i_LeafParent = simplified_skeleton_[i_LeafVertex].nParent;\n'
    '    vec3 pParent = simplified_skeleton_[i_LeafParent].cVert;\n'
    '\t// branch growth direction\n'
    '    vec3 branchDir = (pCurrent - pParent).normalize();\n'
    '\n'
    '\t//generate i-th random leaf\n'
    '\tfor (int i = 0; i < density; ++i)\n'
    '\t{\n'
    '\t\t// leaf base directly at branch tip\n'
    '        double offset = random_float() * radius * 0.5;\n'
    '        vec3 pLeaf = pCurrent - branchDir * offset;\n'
    '\n'
    '\t\t// leaf direction: perpendicular away from branch\n'
    '        vec3 randPerp((random_float()-0.5)/0.5, (random_float()-0.5)/0.5, (random_float()-0.5)/0.5);\n'
    '        randPerp = randPerp.normalize();\n'
    '        randPerp = (randPerp - branchDir * dot(randPerp, branchDir)).normalize();\n'
    '        vec3 dirLeaf = (randPerp * 0.6f + branchDir * 0.4f).normalize();\n'
    '\n'
    '\t\t//generate normal and color vector\n'
    '\t\tvec3 dirParent2Leaf = (pLeaf - pParent).normalize();\n'
    '\t\tvec3 normal = (cross(dirParent2Leaf, dirLeaf)).normalize();\n'
    '\t\t//generate a new leaf\n'
    '\t\tLeaf newleaf;\n'
    '\t\tnewleaf.cPos = pLeaf;\n'
    '\t\tnewleaf.cDir = dirLeaf;\n'
    '\t\t//generate a random normal vector direction\n'
    '        vec3 delta((random_float()-0.5)/0.5, (random_float()-0.5)/0.5, (random_float()-0.5)/0.5);\n'
    '        newleaf.cNormal = (normal + random_float()*delta*0.5).normalize();\n'
    '\t\tnewleaf.pSource = i_LeafVertex;\n'
    '\t\tnewleaf.nLength = BoundingDistance_ * leafsize_Factor;\n'
    '\t\tnewleaf.nRad = newleaf.nLength / 5;\n'
    '\t\tVecLeaves_.push_back(newleaf);\n'
    '\t}\n'
    '\n'
    '\treturn;\n'
    '}'
)
assert old_generate in content, 'ERROR: generate_leaves not found!'
content = content.replace(old_generate, new_generate, 1)
print('Patch C: leaf base attached to branch tip')

# Patch B: elliptic leaf shape
old_func = (
    'bool Skeleton::reconstruct_leaves(SurfaceMesh *mesh) {\n'
    '    if (!add_leaves())\n'
    '        return false;\n'
    '\n'
    '    if (VecLeaves_.empty())\n'
    '        return false;\n'
    '\n'
    '    for (std::size_t i = 0; i < VecLeaves_.size(); i++) {\n'
    '        const Leaf& iLeaf = VecLeaves_[i];\n'
    '        //compute the center and major axis, minor axis of the leaf quad\n'
    '        vec3 pCenter((iLeaf.cPos + (0.5 * iLeaf.cDir * iLeaf.nRad)));\n'
    '        vec3 dirMajor(0.5 * iLeaf.cDir * iLeaf.nLength);\n'
    '        vec3 dirMinor(0.5 * cross(iLeaf.cDir, iLeaf.cNormal)*iLeaf.nRad);\n'
    '        //compute the corner coordinates\n'
    '        const vec3 a = pCenter - dirMajor - dirMinor;\n'
    '        const vec3 b = pCenter + dirMajor - dirMinor;\n'
    '        const vec3 c = pCenter + dirMajor + dirMinor;\n'
    '        const vec3 d = pCenter - dirMajor + dirMinor;\n'
    '        SurfaceMesh::Vertex va = mesh->add_vertex(a);\n'
    '        SurfaceMesh::Vertex vb = mesh->add_vertex(b);\n'
    '        SurfaceMesh::Vertex vc = mesh->add_vertex(c);\n'
    '        SurfaceMesh::Vertex vd = mesh->add_vertex(d);\n'
    '        mesh->add_triangle(va, vb, vc);\n'
    '        mesh->add_triangle(va, vc, vd);\n'
    '    }\n'
    '\n'
    '    return true;\n'
    '}'
)
new_func = (
    'bool Skeleton::reconstruct_leaves(SurfaceMesh *mesh) {\n'
    '    if (!add_leaves())\n'
    '        return false;\n'
    '\n'
    '    if (VecLeaves_.empty())\n'
    '        return false;\n'
    '\n'
    '    const int nSegs = 6;\n'
    '    for (std::size_t i = 0; i < VecLeaves_.size(); i++) {\n'
    '        const Leaf& iLeaf = VecLeaves_[i];\n'
    '        vec3 dir = iLeaf.cDir; vec3 norm = iLeaf.cNormal;\n'
    '        vec3 axisL = dir.normalize() * iLeaf.nLength;\n'
    '        vec3 axisW = cross(dir, norm).normalize() * iLeaf.nRad;\n'
    '        vec3 pBase = iLeaf.cPos;\n'
    '        std::vector<SurfaceMesh::Vertex> leftVerts, rightVerts;\n'
    '        for (int s = 0; s <= nSegs; ++s) {\n'
    '            double t = (double)s / nSegs;\n'
    '            double width = sin(M_PI * t) * (2.0 - 0.3 * t);\n'
    '            vec3 pAlong = pBase + axisL * t;\n'
    '            leftVerts.push_back(mesh->add_vertex(pAlong - axisW * width * 0.5));\n'
    '            rightVerts.push_back(mesh->add_vertex(pAlong + axisW * width * 0.5));\n'
    '        }\n'
    '        for (int s = 0; s < nSegs; ++s) {\n'
    '            mesh->add_triangle(leftVerts[s],   rightVerts[s],   rightVerts[s+1]);\n'
    '            mesh->add_triangle(leftVerts[s],   rightVerts[s+1], leftVerts[s+1]);\n'
    '        }\n'
    '    }\n'
    '    return true;\n'
    '}'
)
assert old_func in content, 'ERROR: reconstruct_leaves not found!'
content = content.replace(old_func, new_func, 1)
print('Patch B: elliptic leaf shape applied')

# Patch D: Least-Squares circle fit for trunk radius
old_bbox = (
    '\t//project the trunk points on the xy plane and get the bounding box\n'
    '\tdouble minX = DBL_MAX;\n'
    '\tdouble maxX = -DBL_MAX;\n'
    '\tdouble minY = DBL_MAX;\n'
    '\tdouble maxY = -DBL_MAX;\n'
    '\tfor (int nP = 0; nP < trunkList.size(); nP++)\n'
    '\t{\n'
    '\t\tif (minX > trunkList[nP].x)\n'
    '\t\t\tminX = trunkList[nP].x;\n'
    '\t\tif (maxX < trunkList[nP].x)\n'
    '\t\t\tmaxX = trunkList[nP].x;\n'
    '\t\tif (minY > trunkList[nP].y)\n'
    '\t\t\tminY = trunkList[nP].y;\n'
    '\t\tif (maxY < trunkList[nP].y)\n'
    '\t\t\tmaxY = trunkList[nP].y;\n'
    '\t}\n'
    '\n'
    '\t//assign the raw radius value and return\n'
    '    TrunkRadius_ = std::max((maxX - minX), (maxY - minY)) / 2.0;\n'
)
new_bbox = (
    '\t// Patch D: Least-Squares circle fit instead of bounding box\n'
    '\tdouble minX = DBL_MAX, maxX = -DBL_MAX;\n'
    '\tdouble minY = DBL_MAX, maxY = -DBL_MAX;\n'
    '\tfor (int nP = 0; nP < (int)trunkList.size(); nP++) {\n'
    '\t\tif (minX > trunkList[nP].x) minX = trunkList[nP].x;\n'
    '\t\tif (maxX < trunkList[nP].x) maxX = trunkList[nP].x;\n'
    '\t\tif (minY > trunkList[nP].y) minY = trunkList[nP].y;\n'
    '\t\tif (maxY < trunkList[nP].y) maxY = trunkList[nP].y;\n'
    '\t}\n'
    '\tdouble cx = 0.0, cy = 0.0;\n'
    '\tfor (int nP = 0; nP < (int)trunkList.size(); nP++) {\n'
    '\t\tcx += trunkList[nP].x; cy += trunkList[nP].y;\n'
    '\t}\n'
    '\tcx /= (int)trunkList.size(); cy /= (int)trunkList.size();\n'
    '\tdouble r  = std::max((maxX - minX), (maxY - minY)) / 2.0;\n'
    '\tdouble r_max = r * 2.0;\n'
    '\tif ((int)trunkList.size() >= 1000) {\n'
    '\t\tfor (int iter = 0; iter < 100; ++iter) {\n'
    '\t\t\tdouble dCx=0, dCy=0, dR=0;\n'
    '\t\t\tdouble A00=0,A01=0,A02=0,A11=0,A12=0,A22=0,b0=0,b1=0,b2=0;\n'
    '\t\t\tfor (int nP = 0; nP < (int)trunkList.size(); nP++) {\n'
    '\t\t\t\tdouble dx=trunkList[nP].x-cx, dy=trunkList[nP].y-cy;\n'
    '\t\t\t\tdouble dist=std::sqrt(dx*dx+dy*dy);\n'
    '\t\t\t\tif (dist<1e-10) continue;\n'
    '\t\t\t\tdouble res=dist-r, Jcx=-dx/dist, Jcy=-dy/dist, Jr=-1.0;\n'
    '\t\t\t\tA00+=Jcx*Jcx; A01+=Jcx*Jcy; A02+=Jcx*Jr;\n'
    '\t\t\t\tA11+=Jcy*Jcy; A12+=Jcy*Jr;  A22+=Jr*Jr;\n'
    '\t\t\t\tb0+=Jcx*res;  b1+=Jcy*res;  b2+=Jr*res;\n'
    '\t\t\t}\n'
    '\t\t\tdouble det=A00*(A11*A22-A12*A12)-A01*(A01*A22-A12*A02)+A02*(A01*A12-A11*A02);\n'
    '\t\t\tif (std::abs(det)<1e-14) break;\n'
    '\t\t\tdCx=(b0*(A11*A22-A12*A12)-A01*(b1*A22-A12*b2)+A02*(b1*A12-A11*b2))/det;\n'
    '\t\t\tdCy=(A00*(b1*A22-A12*b2)-b0*(A01*A22-A12*A02)+A02*(A01*b2-b1*A02))/det;\n'
    '\t\t\tdR=(A00*(A11*b2-b1*A12)-A01*(A01*b2-b1*A02)+b0*(A01*A12-A11*A02))/det;\n'
    '\t\t\tcx-=dCx; cy-=dCy; r-=dR;\n'
    '\t\t\tif (r<0) r=std::abs(r);\n'
    '\t\t\tif (r>r_max) { r=r_max; break; }\n'
    '\t\t\tif (std::abs(dCx)+std::abs(dCy)+std::abs(dR)<1e-8) break;\n'
    '\t\t}\n'
    '\t}\n'
    '    TrunkRadius_ = r;\n'
)
assert old_bbox in content, 'ERROR: BBox not found!'
content = content.replace(old_bbox, new_bbox, 1)
print('Patch D: least-squares circle fit applied')

with open(SKEL, 'w') as f:
    f.write(content)
print('All patches applied successfully')
PYEOF

mkdir -p $SRCDIR/Release && cd $SRCDIR/Release
cmake -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_FLAGS="-w" .. 2>&1 | tail -4
echo "Compiling AdTree (3-5 min)..."
make -j$(nproc) 2>&1 | tail -10

EXEC=$(find $SRCDIR/Release/bin -name 'AdTree' -type f 2>/dev/null | head -1)
if [ -n "$EXEC" ]; then
    echo "AdTree compiled: $EXEC"
    echo "$EXEC" > /content/adtree_exec_path.txt
else
    echo "ERROR: compilation failed"
    make -j$(nproc) 2>&1 | grep 'error:' | head -5
fi


## Step 3: Configuration
Set scan type and filtering options here.

In [ ]:
import os, glob

# --- Input file ---
# The pipeline auto-detects your uploaded file.
# Supported: .las .laz .ply .txt .csv
# To set manually: INPUT_FILE = '/content/your_file.laz'
EXTENSIONS = ['*.las', '*.laz', '*.ply', '*.txt', '*.csv']
found = []
for ext in EXTENSIONS:
    found += glob.glob(f'/content/{ext}')
found = [f for f in found if not os.path.basename(f).startswith('.')]
assert found, (
    'No point cloud file found. '
    'Upload a .las / .laz / .ply / .txt / .csv file to Colab.'
)
INPUT_FILE = found[0]
if len(found) > 1:
    print(f'Multiple files found - using: {os.path.basename(INPUT_FILE)}')
    print(f'All found: {[os.path.basename(f) for f in found]}')
    print('To use a different file: INPUT_FILE = "/content/your_file.laz"')
else:
    print(f'Input file: {INPUT_FILE}')

# --- PointTree settings ---
SCAN_TYPE = 'TLS'       # 'TLS' for terrestrial scan, 'ULS' for drone/aerial scan
CLOUD_ID  = os.path.splitext(os.path.basename(INPUT_FILE))[0]

# --- Output directories ---
POINTTREE_OUT = '/content/tree_clouds'
ADTREE_OUT    = '/content/forest_output'
FINAL_ZIP     = f'/content/{CLOUD_ID}_results.zip'

# --- AdTree noise filtering (optional) ---
ENABLE_NOISE_FILTER = False
NB_NEIGHBORS        = 20
STD_RATIO           = 2.0
VOXEL_SIZE          = 0.05   # meters

# --- Leaf filtering ---
FACES_PER_LEAF = 12          # 6 segments x 2 triangles (elliptic leaf shape)
KEEP_RATIOS    = [0.6, 0.3]  # two filtered versions per tree

os.makedirs(POINTTREE_OUT, exist_ok=True)
os.makedirs(ADTREE_OUT, exist_ok=True)
print('Configuration done.')
print(f'  Scan type : {SCAN_TYPE}')
print(f'  Cloud ID  : {CLOUD_ID}')


## Step 4: Load point cloud

In [ ]:
import pointtorch
import numpy as np

print(f'Loading {INPUT_FILE} ...')
point_cloud = pointtorch.read(INPUT_FILE)

print(f'{len(point_cloud):,} points loaded')
print(f'  Columns : {list(point_cloud.columns)}')
print(f'  X range : {point_cloud["x"].min():.2f} - {point_cloud["x"].max():.2f}')
print(f'  Y range : {point_cloud["y"].min():.2f} - {point_cloud["y"].max():.2f}')
print(f'  Z range : {point_cloud["z"].min():.2f} - {point_cloud["z"].max():.2f}')


## Step 5: Tree instance segmentation (PointTree)

In [ ]:
import sys, io, os, numpy as np
from pointtree.instance_segmentation import TreeXAlgorithm, TreeXPresetTLS, TreeXPresetULS

# Load preset
if SCAN_TYPE == 'TLS':
    preset = dict(TreeXPresetTLS())
else:
    preset = dict(TreeXPresetULS())
preset['visualization_folder'] = None

algorithm = TreeXAlgorithm(**preset)

# Extract XYZ and optional intensity
xyz = point_cloud[['x', 'y', 'z']].to_numpy()
intensities = None
if 'intensity' in point_cloud.columns:
    intensities = point_cloud['intensity'].to_numpy()
    print('Intensity values found and used')
else:
    print('No intensity column - continuing without')

print(f'Running TreeX segmentation ({SCAN_TYPE} preset) on {len(xyz):,} points...')

# Colab-compatible stdout wrapper (avoids fileno() errors)
class ColabStdout(io.RawIOBase):
    def write(self, b):
        if isinstance(b, bytes):
            sys.__stdout__.write(b.decode('utf-8', errors='replace'))
        else:
            sys.__stdout__.write(b)
        return len(b)
    def fileno(self):
        return sys.__stdout__.fileno()

_old_stdout = sys.stdout
_old_stderr = sys.stderr
sys.stdout = ColabStdout()
sys.stderr = ColabStdout()
try:
    instance_ids, trunk_positions, trunk_diameters = algorithm(
        xyz, intensities=intensities, point_cloud_id=CLOUD_ID
    )
finally:
    sys.stdout = _old_stdout
    sys.stderr = _old_stderr

unique_ids = np.unique(instance_ids)
tree_ids   = unique_ids[unique_ids >= 0]
n_noise    = int(np.sum(instance_ids == -1))

print(f'Segmentation result:')
print(f'  Trees detected  : {len(tree_ids)}')
print(f'  Non-tree points : {n_noise:,}')
print(f'  Trunk positions : {len(trunk_positions)}')
if len(trunk_diameters) > 0:
    print(f'  Mean trunk diam : {np.mean(trunk_diameters):.3f} m')


## Step 6: Save PointTree results
Saves the labeled full cloud, individual tree clouds, assembly JSON and summary CSV.

In [ ]:
import os, json
import pandas as pd
import numpy as np

# Full cloud with instance_id label
point_cloud['instance_id'] = instance_ids
base_cols  = ['x', 'y', 'z', 'instance_id']
extra_cols = [c for c in ['intensity', 'r', 'g', 'b'] if c in point_cloud.columns]
save_cols  = base_cols + extra_cols

labeled_path = os.path.join(POINTTREE_OUT, f'{CLOUD_ID}_labeled.laz')
point_cloud.to(labeled_path, columns=save_cols)
print(f'Labeled cloud saved: {labeled_path}')

# Individual tree clouds
trees_dir = os.path.join(POINTTREE_OUT, 'individual_trees')
os.makedirs(trees_dir, exist_ok=True)
save_cols_tree = [c for c in save_cols if c != 'instance_id']

scene_origin = xyz.mean(axis=0)
summary = []

for i, tree_id in enumerate(tree_ids):
    mask      = instance_ids == tree_id
    tree_cloud = point_cloud[mask].copy()
    tree_xyz   = xyz[mask]

    bbox_min = tree_xyz.min(axis=0)
    bbox_max = tree_xyz.max(axis=0)
    centroid = tree_xyz.mean(axis=0)

    trunk_x = float(trunk_positions[i, 0]) if i < len(trunk_positions) else float(centroid[0])
    trunk_y = float(trunk_positions[i, 1]) if i < len(trunk_positions) else float(centroid[1])
    trunk_z = float(bbox_min[2])
    diam    = float(trunk_diameters[i]) if i < len(trunk_diameters) else None

    out_path = os.path.join(trees_dir, f'tree_{tree_id:04d}.laz')
    tree_cloud.to(out_path, columns=save_cols_tree)

    meta = {
        'tree_id':      int(tree_id),
        'file':         f'tree_{tree_id:04d}.laz',
        'n_points':     int(mask.sum()),
        'trunk_x':      round(trunk_x, 4),
        'trunk_y':      round(trunk_y, 4),
        'trunk_z':      round(trunk_z, 4),
        'trunk_diam_m': round(diam, 4) if diam else None,
        'centroid_x':   round(float(centroid[0]), 4),
        'centroid_y':   round(float(centroid[1]), 4),
        'centroid_z':   round(float(centroid[2]), 4),
        'bbox_min':     [round(float(v), 4) for v in bbox_min],
        'bbox_max':     [round(float(v), 4) for v in bbox_max],
        'height_m':     round(float(bbox_max[2] - bbox_min[2]), 3),
        'offset_from_scene_origin': [
            round(float(trunk_x - scene_origin[0]), 4),
            round(float(trunk_y - scene_origin[1]), 4),
            round(float(trunk_z - scene_origin[2]), 4),
        ]
    }
    summary.append(meta)

    with open(os.path.join(trees_dir, f'tree_{tree_id:04d}_meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)

# Assembly JSON
assembly = {
    'cloud_id':     CLOUD_ID,
    'scene_origin': [round(float(v), 4) for v in scene_origin],
    'n_trees':      len(tree_ids),
    'trees':        summary
}
assembly_path = os.path.join(POINTTREE_OUT, f'{CLOUD_ID}_assembly.json')
with open(assembly_path, 'w') as f:
    json.dump(assembly, f, indent=2)

# Summary CSV
df = pd.DataFrame(summary)
csv_path = os.path.join(POINTTREE_OUT, f'{CLOUD_ID}_tree_summary.csv')
df.to_csv(csv_path, index=False)

print(f'{len(tree_ids)} individual tree clouds saved: {trees_dir}')
print(f'Assembly JSON : {assembly_path}')
print(f'Summary CSV   : {csv_path}')
print()
print(df[['tree_id', 'n_points', 'height_m', 'trunk_diam_m', 'trunk_x', 'trunk_y']].to_string(index=False))


## Step 7: AdTree branch and leaf reconstruction

In [ ]:
import laspy
import open3d as o3d
import numpy as np
import subprocess, os, shutil

with open('/content/adtree_exec_path.txt') as f:
    ADTREE_BIN = f.read().strip()
print(f'AdTree binary: {ADTREE_BIN}')

def filter_leaves_obj(leaves_in, leaves_out, keep_ratio, faces_per_leaf=12):
    """Filter a leaves OBJ file to keep_ratio fraction of leaves."""
    with open(leaves_in) as f:
        lines = f.readlines()
    vertices = [l for l in lines if l.startswith('v ')]
    faces    = [l for l in lines if l.startswith('f ')]
    header   = [l for l in lines if not l.startswith('v ') and not l.startswith('f ')]
    n_leaves = len(faces) // faces_per_leaf
    n_keep   = max(1, int(n_leaves * keep_ratio))
    np.random.seed(42)
    keep_idx = np.sort(np.random.choice(n_leaves, n_keep, replace=False))
    kept_faces = []
    for idx in keep_idx:
        kept_faces.extend(faces[idx*faces_per_leaf : (idx+1)*faces_per_leaf])
    used_v = set()
    for face in kept_faces:
        for token in face.strip().split()[1:]:
            used_v.add(int(token.split('/')[0]) - 1)
    old_to_new = {}
    new_verts  = []
    for old_idx in sorted(used_v):
        old_to_new[old_idx] = len(new_verts) + 1
        new_verts.append(vertices[old_idx])
    new_faces = []
    for face in kept_faces:
        parts   = face.strip().split()
        new_idx = [str(old_to_new[int(p.split('/')[0]) - 1]) for p in parts[1:]]
        new_faces.append('f ' + ' '.join(new_idx) + '\n')
    with open(leaves_out, 'w') as f:
        f.writelines(header)
        f.writelines(new_verts)
        f.writelines(new_faces)

# Process each tree from PointTree output
for tree in summary:
    tid      = f'tree_{tree["tree_id"]:04d}'
    laz_path = os.path.join(trees_dir, tree['file'])
    tree_out = os.path.join(ADTREE_OUT, tid)
    os.makedirs(tree_out, exist_ok=True)

    print(f'{tid}  ({tree["n_points"]:,} pts  H={tree["height_m"]:.1f}m)')

    # Load LAZ
    las = laspy.read(laz_path)
    pts = np.vstack([las.x, las.y, las.z]).T
    print(f'  {len(pts):,} points loaded')

    # Optional noise filtering
    if ENABLE_NOISE_FILTER:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(pts)
        pcd, _ = pcd.remove_statistical_outlier(
            nb_neighbors=NB_NEIGHBORS, std_ratio=STD_RATIO)
        pcd = pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)
        pts = np.asarray(pcd.points)
        print(f'  After filtering: {len(pts):,} points')

    # Save point cloud as PLY
    pcd_save = o3d.geometry.PointCloud()
    pcd_save.points = o3d.utility.Vector3dVector(pts)
    o3d.io.write_point_cloud(os.path.join(tree_out, f'{tid}_pointcloud.ply'), pcd_save)

    # Write XYZ for AdTree
    xyz_file = os.path.join(tree_out, f'{tid}.xyz')
    np.savetxt(xyz_file, pts, fmt='%.6f')

    # Run AdTree
    adtree_raw = os.path.join(tree_out, 'adtree_raw')
    os.makedirs(adtree_raw, exist_ok=True)
    proc = subprocess.run(
        [ADTREE_BIN, xyz_file, adtree_raw, '-skeleton'],
        capture_output=True, text=True, timeout=900
    )

    src_br = os.path.join(adtree_raw, f'{tid}_branches.obj')
    src_lv = os.path.join(adtree_raw, f'{tid}_leaves.obj')
    src_sk = os.path.join(adtree_raw, f'{tid}_skeleton.ply')

    if not os.path.exists(src_br):
        print(f'  ERROR: AdTree failed - skipping')
        print(proc.stderr[-300:])
        tree['adtree_success'] = False
        continue
    tree['adtree_success'] = True
    tree['output_dir']     = tree_out

    shutil.copy(src_br, os.path.join(tree_out, f'{tid}_branches.obj'))
    shutil.copy(src_sk, os.path.join(tree_out, f'{tid}_skeleton.ply'))

    leaves_orig = os.path.join(tree_out, f'{tid}_leaves.obj')
    shutil.copy(src_lv, leaves_orig)

    for ratio in KEEP_RATIOS:
        ratio_str = str(ratio).replace('.', '')
        filter_leaves_obj(
            leaves_orig,
            os.path.join(tree_out, f'{tid}_leaves_filtered{ratio_str}.obj'),
            ratio, FACES_PER_LEAF
        )

    print(f'  Done -> {tree_out}')

n_ok = sum(1 for t in summary if t.get('adtree_success', False))
print(f'\nAdTree pipeline complete: {n_ok}/{len(summary)} trees successful')


## Step 8: Merge all trees into forest-level files

In [ ]:
import open3d as o3d
import numpy as np
import os

forest_out = os.path.join(ADTREE_OUT, 'forest')
os.makedirs(forest_out, exist_ok=True)

def translate_obj(obj_path, offset):
    """Read OBJ, shift all vertices by offset, return lines."""
    with open(obj_path) as f:
        lines = f.readlines()
    result = []
    for line in lines:
        if line.startswith('v '):
            parts = line.strip().split()
            x = float(parts[1]) + offset[0]
            y = float(parts[2]) + offset[1]
            z = float(parts[3]) + offset[2]
            result.append(f'v {x:.6f} {y:.6f} {z:.6f}\n')
        else:
            result.append(line)
    return result

def merge_objs(obj_paths_and_offsets, out_path):
    """Merge multiple OBJ files with per-file offsets into one OBJ."""
    all_lines     = []
    vertex_offset = 0
    for obj_path, offset in obj_paths_and_offsets:
        if not os.path.exists(obj_path):
            continue
        lines    = translate_obj(obj_path, offset)
        vertices = [l for l in lines if l.startswith('v ')]
        faces    = [l for l in lines if l.startswith('f ')]
        new_faces = []
        for face in faces:
            parts   = face.strip().split()
            new_idx = [str(int(p.split('/')[0]) + vertex_offset) for p in parts[1:]]
            new_faces.append('f ' + ' '.join(new_idx) + '\n')
        all_lines.extend(vertices)
        all_lines.extend(new_faces)
        vertex_offset += len(vertices)
    with open(out_path, 'w') as f:
        f.writelines(all_lines)
    size_mb = os.path.getsize(out_path) / 1024 / 1024
    print(f'  {os.path.basename(out_path):50s} {size_mb:.1f} MB')

success_trees = [t for t in summary if t.get('adtree_success', False)]
print(f'Merging {len(success_trees)} trees...')

def get_offset(tree):
    return tree.get('offset_from_scene_origin', [0, 0, 0])

# Merge point clouds
all_pts = []
for t in success_trees:
    ply = os.path.join(t['output_dir'], f'tree_{t["tree_id"]:04d}_pointcloud.ply')
    if os.path.exists(ply):
        pcd = o3d.io.read_point_cloud(ply)
        all_pts.append(np.asarray(pcd.points) + np.array(get_offset(t)))
if all_pts:
    merged = np.vstack(all_pts)
    pcd_f  = o3d.geometry.PointCloud()
    pcd_f.points = o3d.utility.Vector3dVector(merged)
    o3d.io.write_point_cloud(os.path.join(forest_out, 'forest_pointcloud.ply'), pcd_f)
    print(f'  forest_pointcloud.ply  ({len(merged):,} points)')

# Merge OBJ files
merge_objs(
    [(os.path.join(t['output_dir'], f'tree_{t["tree_id"]:04d}_branches.obj'), get_offset(t))
     for t in success_trees],
    os.path.join(forest_out, 'forest_branches.obj')
)
merge_objs(
    [(os.path.join(t['output_dir'], f'tree_{t["tree_id"]:04d}_leaves.obj'), get_offset(t))
     for t in success_trees],
    os.path.join(forest_out, 'forest_leaves.obj')
)
for ratio in KEEP_RATIOS:
    ratio_str = str(ratio).replace('.', '')
    merge_objs(
        [(os.path.join(t['output_dir'], f'tree_{t["tree_id"]:04d}_leaves_filtered{ratio_str}.obj'), get_offset(t))
         for t in success_trees],
        os.path.join(forest_out, f'forest_leaves_filtered{ratio_str}.obj')
    )

print('Forest merge complete.')


## Step 9: Create output ZIP

In [ ]:
import zipfile, os

print(f'Creating ZIP: {FINAL_ZIP}')

with zipfile.ZipFile(FINAL_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    # PointTree results
    for root, dirs, files in os.walk(POINTTREE_OUT):
        for file in files:
            full_path = os.path.join(root, file)
            arcname   = os.path.join('pointtree', os.path.relpath(full_path, POINTTREE_OUT))
            zf.write(full_path, arcname)
    # AdTree results
    for root, dirs, files in os.walk(ADTREE_OUT):
        for file in files:
            full_path = os.path.join(root, file)
            arcname   = os.path.join('adtree', os.path.relpath(full_path, ADTREE_OUT))
            zf.write(full_path, arcname)

size_mb = os.path.getsize(FINAL_ZIP) / 1024 / 1024
print(f'ZIP created: {FINAL_ZIP}  ({size_mb:.1f} MB)')

# Show top-level structure
with zipfile.ZipFile(FINAL_ZIP) as zf:
    names = zf.namelist()
print(f'{len(names)} files in ZIP')
dirs_seen = set()
for name in sorted(names):
    d = '/'.join(name.split('/')[:2])
    if d not in dirs_seen:
        print(f'  {d}/')
        dirs_seen.add(d)


## Step 10: Download results

In [ ]:
from google.colab import files
files.download(FINAL_ZIP)
print(f'Download started: {os.path.basename(FINAL_ZIP)}')
